In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round, avg

spark = SparkSession.builder \
    .appName("TugasMandiriETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/09/24 11:02:50 WARN Utils: Your hostname, raam resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/24 11:02:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 11:02:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 11:02:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
print("Versi Spark:", spark.version)

Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [13]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Jumlah baris Transaksi : {df_transaksi.count()}")
print(f"Jumlah baris Produk    : {df_produk.count()}")
print(f"Jumlah baris Ulasan    : {df_ulasan.count()}")

print("Data Transaksi")
df_transaksi.printSchema()

print("Data Produk")
df_produk.printSchema()

print("Data Ulasan")
df_ulasan.printSchema()


Jumlah baris Transaksi : 5000
Jumlah baris Produk    : 30
Jumlah baris Ulasan    : 3500
Data Transaksi
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [12]:
df_gabung = df_transaksi.join(df_produk, on="product_id", how="inner")\
                              .join(df_ulasan, on="order_id", how="left")

df_transformed = df_gabung.withColumn("total_pendapatan", col("unit_terjual") + col("harga"))

df_transformed.show(5)
                              

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           25002|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|          250006|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|           25004|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|           75004|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|           75006|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [17]:
df_cleaned = df_transformed.withColumn("ada_ulasan", col("rating").isNotNull())

df_cleaned = df_cleaned.fillna({"rating": 0})

df_cleaned.select("order_id", "product_id", "rating", "ada_ulasan").show(10)

+--------+----------+------+----------+
|order_id|product_id|rating|ada_ulasan|
+--------+----------+------+----------+
|     TX0|        21|     1|      true|
|     TX1|         8|     0|     false|
|     TX2|        25|     2|      true|
|     TX3|         3|     5|      true|
|     TX4|        19|     0|     false|
|     TX5|        10|     5|      true|
|     TX6|        26|     4|      true|
|     TX7|         4|     0|     false|
|     TX8|         7|     5|      true|
|     TX9|        30|     0|     false|
+--------+----------+------+----------+
only showing top 10 rows



In [18]:
hdfs_path = "hdfs://localhost:9000/user/hadoop/output_tugas_mandiri"

df_cleaned.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

df_hdfs_verify = spark.read.parquet(hdfs_path)

print("Jumlah baris data di HDFS : {df_hdfs_verify.count()}")
print("verifikasi jumlah baris yang cocok : {df_hdfs_verify.count() == df_cleaned.count()}")

!hdfs dfs -ls /user/hadoop/output_tugas_mandiri

Jumlah baris data di HDFS : {df_hdfs_verify.count()}
verifikasi jumlah baris yang cocok : {df_hdfs_verify.count() == df_cleaned.count()}
Found 6 items
-rw-r--r--   3 raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/_SUCCESS
drwxr-xr-x   - raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/kategori=Elektronik
drwxr-xr-x   - raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/kategori=Fashion
drwxr-xr-x   - raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/kategori=Kesehatan
drwxr-xr-x   - raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/kategori=Makanan
drwxr-xr-x   - raamxh supergroup          0 2026-09-24 11:35 /user/hadoop/output_tugas_mandiri/kategori=Rumah Tangga


In [20]:
df_insight = df_cleaned.groupBy("kategori").agg( count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, True)).alias("jumlah_berulasan")).withColumn("persentase_ulasan",
    round((col("jumlah_berulasan") / col("total_transaksi")) * 100, 2)).orderBy("persentase_ulasan")

df_insight.show()

+------------+---------------+----------------+-----------------+
|    kategori|total_transaksi|jumlah_berulasan|persentase_ulasan|
+------------+---------------+----------------+-----------------+
|     Makanan|            536|             369|            68.84|
|   Kesehatan|            961|             662|            68.89|
|     Fashion|           1035|             726|            70.14|
|Rumah Tangga|            815|             575|            70.55|
|  Elektronik|           1653|            1168|            70.66|
+------------+---------------+----------------+-----------------+



Kategori dengan persentase transaksi berulasan paling rendah menunjukkan tingkat partisipasi dan keterlibatan (engagement) pasca-pembelian yang paling minim dari pelanggan. Informasi ini sangat krusial bagi tim marketing untuk merancang program insentif spesifik (seperti penawaran poin loyalitas, kupon diskon, atau pengingat otomatis via pesan singkat/email) guna mendorong pembeli memberikan ulasan. Peningkatan jumlah ulasan akan memperkuat social proof produk serta mendongkrak tingkat konversi penjualan di masa mendatang.

In [4]:
!ls

 biodata.txt
 contoh_lokal.txt
 data_gabungan_bersih.csv
 data_pelanggan.json
 data_transaksi_besar.csv
 data_transaksi_besar_parquet
 data_transaksi_ecommerce.csv
 data_transaksi_partisi_kota
 hasil_unduh_dari_hdfs.txt
 latihan_partisi_membership
 Modul_Praktikum_BigData_Pertemuan2.ipynb
 Modul_Praktikum_BigData_Pertemuan3.ipynb
 Modul_Praktikum_BigData_Pertemuan4.ipynb
 Modul_Praktikum_BigData_Pertemuan5.ipynb
 Modul_Praktikum_BigData_Pertemuan6.ipynb
 Modul_Praktikum_BigData_Pertemuan7.ipynb
 ringkasan_kota_kategori.csv
 transaksi_magelang.csv
 transaksi_semarang.csv
 transaksi_september_2026.csv
 transaksi_tugas5.csv
 transaksi_yogyakarta.csv
'Tugas_2505060025_Ramudya Fuji Pratama.ipynb'
'Tugas4_2505060025_Ramudya Fuji Pratama.ipynb'
'Tugas5_2505060025_Ramudya Fuji Pratama.ipynb'
'Tugas6_2505060025_Ramudya Fuji Pratama.ipynb'
 tugas6_produk.json
 tugas6_transaksi.csv
 tugas6_ulasan.csv
